<div align="center">
  <img src="https://raw.githubusercontent.com/harishs-io/lateral_flowimmunoassay_quant/main/assets/lfia_cover_image.png" width="100%">
  <h1>🧬 LFIA Signal Quantification & LoD Calculation</h1>
</div>

---

** Image Requirements:**
* **Assay Type:** Optimized for AuNP (Colloidal Gold) based LFIAs, with each image containing exactly one assay strip.
* **Naming Convention:** Use `[concentration]_[replicate].[ext]` (e.g., `0.1_1.jpg`, `1_2.png`) for automatic replicate grouping and statistical analysis.
* **Supported Formats:** Any standard image format is supported for quantitative processing and LoD calculation.

---

**Instructions:**
1. Run the **Environment Setup** cell to load the pipeline.
2. Run the **Upload & Analyze** cell to insert your image(s) and extract the signal.
3. Run the **Export** cell to download your data.

In [ ]:
#@title ⚙️ 1. Environment Setup { display-mode: "form" }
#@markdown Run this cell once to configure the workspace. Installation logs will be displayed below.

!git clone https://github.com/harishs-io/lateral_flowimmunoassay_quant.git
%cd lateral_flowimmunoassay_quant
!pip install ultralytics opencv-python matplotlib scipy pandas numpy

import os
import sys
sys.path.append('.')
from google.colab import files
from IPython.display import Image, display

print("\n✅ Pipeline environment successfully loaded! Move to Step 2.")

In [ ]:
#@title 🔬 2. Upload Image & Run Pipeline { display-mode: "form" }
#@markdown Adjust the detection parameters and choose whether to fine-tune the model with your images.

yolo_confidence = 0.20 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
fine_tune_yolo = "No" #@param ["Yes", "No"]

import shutil
import builtins
import glob
import os
import io
import re
import base64
from contextlib import redirect_stdout
from IPython.display import clear_output, Image, display, HTML

# Map the UI dropdown to the script's input prompt
auto_response = 'y' if fine_tune_yolo == "Yes" else 'n'
builtins.input = lambda prompt: auto_response

def get_b64(path):
    with open(path, "rb") as f: return base64.b64encode(f.read()).decode()

print("👇 Click 'Choose Files' below to insert your assay image(s):")
uploaded = files.upload()

if uploaded:
    clear_output()
    print(f"✅ Successfully uploaded {len(uploaded)} image(s)!")
    print("⏳ Processing images through YOLOv8 and extracting chromatograms... (This may take a moment)\n")
    
    base_dir = "colab_run"
    sub_dir = os.path.join(base_dir, "assay_01")
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    os.makedirs(sub_dir, exist_ok=True)
    
    for image_name in uploaded.keys():
        shutil.move(image_name, os.path.join(sub_dir, image_name))
    
    try:
        from src.lfia_analyzer import run_pipeline
        
        trap = io.StringIO()
        with redirect_stdout(trap):
            run_pipeline(base_dir=base_dir, dataset_dir="dataset", weights_path="weights/best.pt", conf_thresh=yolo_confidence)
        
        print("✅ Pipeline complete! Targets localized and integrated.\n")
        
        # Extract LoD from the summary text file
        summary_files = glob.glob(f"{sub_dir}/Results/*_Summary_Results.txt")
        lod_text = "LoD: Insufficient points or variance to calculate limits."
        if summary_files:
            with open(summary_files[0], 'r') as sf:
                for line in sf:
                    if "LoD" in line and "*" in line:
                        lod_text = line.replace("*", "").strip()
                        break
                        
        print("---")
        print(f"🎯 {lod_text}")
        print("---")
        
        # Display standard curve at the top
        std_curve = glob.glob(f"{sub_dir}/Results/*_Standard_Curve_Plot.png")
        if std_curve:
            display(Image(filename=std_curve[0], width=550))
            print("\n")
        
        # Display all images in a 2-column HTML grid
        txt_files = glob.glob(f"{sub_dir}/Results/**/*_results.txt", recursive=True)
        txt_files.sort()
        
        html_content = '<div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px;">'
        
        for txt_file in txt_files:
            with open(txt_file, 'r') as f:
                content = f.read()
                
            conc_match = re.search(r'Concentration:\s*(.*)', content)
            t_match = re.search(r'Test \(T\) Area:\s*([0-9.]+)', content)
            c_match = re.search(r'Control \(C\) Area:\s*([0-9.]+)', content)
            tc_match = re.search(r'T/C Ratio:\s*([0-9.]+)', content)
            
            conc = conc_match.group(1).strip() if conc_match else "Unknown"
            t_val = t_match.group(1) if t_match else "N/A"
            c_val = c_match.group(1) if c_match else "N/A"
            tc_val = tc_match.group(1) if tc_match else "N/A"
            
            plot_file = txt_file.replace('_results.txt', '_Analysis_Plot.png')
            img_html = ""
            if os.path.exists(plot_file):
                img_html = f'<img src="data:image/png;base64,{get_b64(plot_file)}" style="width:100%; border-radius:6px; margin-top:8px; border: 1px solid #eee;">'
                
            html_content += f'''
            <div style="background: #f8f9fa; padding: 12px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); border: 1px solid #e0e0e0;">
                <div style="font-family: sans-serif; font-size: 13px; color: #202124;">
                    <b>🔬 {conc}</b> | T: {t_val} | C: {c_val} | <b>T/C: {tc_val}</b>
                </div>
                {img_html}
            </div>
            '''
            
        html_content += '</div>'
        display(HTML(html_content))
            
    except Exception as e:
        print(f"\n⚠️ Pipeline Error: {e}")
else:
    print("⚠️ No file uploaded. Please run the cell again.")

In [ ]:
#@title 📊 3. Export Data & Reports { display-mode: "form" }
#@markdown Zips the generated `Results/` folder (including cropped images, text reports, and plots) and downloads it to your machine.

try:
    shutil.make_archive("LFIA_Results", 'zip', "colab_run/assay_01/Results")
    print("✅ Results packaged. Downloading 'LFIA_Results.zip'...")
    files.download('LFIA_Results.zip')
except Exception as e:
    print("⚠️ Error: Please run Step 2 first to generate the results.")